# 🌙 08. Spatial Grid Uniform Keypoint Distribution Engine

**Mission Context**: Eliminating keypoint clustering on single high-contrast crater rims to enforce uniform spatial coverage across the lunar tile.  
**Objectives**:
- Divide image into $8 \times 8$ spatial bins.
- Select top scoring keypoints per grid cell with minimum intra-cell distance constraints.
- Calculate **Coverage Score**, **Uniform Distribution Score**, and **Grid Occupancy**.
- Visualize spatial distributions before and after grid regularization.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.distribution import UniformGridDistributor

config = load_config()
raw_ref = np.load("outputs/features/features_reference.npz")
kpts_raw = raw_ref["keypoints"]
scores_raw = raw_ref["scores"]
desc_raw = raw_ref["descriptors"]

distributor = UniformGridDistributor(grid_rows=8, grid_cols=8, max_points_per_cell=16, min_distance_px=10)
dist_res = distributor.distribute(kpts_raw, scores_raw, desc_raw, (512, 512))

print(f"Original Points: {len(kpts_raw)} -> Distributed Points: {len(dist_res['keypoints'])}")
print(f"Coverage Score: {dist_res['coverage_score']}%")
print(f"Uniform Distribution Score: {dist_res['uniform_score']}")
print(f"Grid Cell Occupancy: {dist_res['grid_occupancy'] * 100:.1f}%")


In [ ]:
# Visualize Before vs. After Spatial Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Before
axes[0].scatter(kpts_raw[:, 0], kpts_raw[:, 1], c='red', s=12, alpha=0.6)
axes[0].set_xlim(0, 512)
axes[0].set_ylim(512, 0)
axes[0].set_title(f"Before Uniform Distribution (Clustered, N={len(kpts_raw)})", fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)

# After with 8x8 Grid overlay
kpts_dist = dist_res['keypoints']
axes[1].scatter(kpts_dist[:, 0], kpts_dist[:, 1], c='green', s=15, alpha=0.8)
for i in range(1, 8):
    axes[1].axvline(i * 64, color='blue', linestyle=':', alpha=0.4)
    axes[1].axhline(i * 64, color='blue', linestyle=':', alpha=0.4)

axes[1].set_xlim(0, 512)
axes[1].set_ylim(512, 0)
axes[1].set_title(f"After Grid Regularization (Coverage: {dist_res['coverage_score']}%, N={len(kpts_dist)})", fontweight='bold')

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/08_uniform_distribution.png", dpi=300)
plt.show()
